In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

In [ ]:
#Export training and validate from ML model
file_path = "/N/lustre/project/proj-212/Sumit/Project CHa/XGB Linear/training_data.csv"
validate_path = "/N/lustre/project/proj-212/Sumit/Project CHa/XGB Linear/validate_data.csv"
df = pd.read_csv(file_path)
validate = pd.read_csv(validate_path)


In [ ]:
# Step 1: Compute Relative Bias
df_train['RB'] = (df_train[predicted_col] - df_train[observed_col]) / df_train[observed_col]
df_val['RB'] = (df_val[predicted_col] - df_val[observed_col]) / df_val[observed_col]


In [ ]:
# NSE function
def nse(preds, obs):
    return 1 - np.sum((preds - obs)**2) / np.sum((obs - np.mean(obs))**2)

# Apply correction per bin using fitted models
def apply_tree_correction(df, models_dict):
    corrected = []
    for _, row in df.iterrows():
        bin_label = row['TreeBin']
        if pd.notnull(bin_label) and bin_label in models_dict:
            model = models_dict[bin_label]
            x = pd.DataFrame({predicted_col: [row[predicted_col]]})
            corrected_value = model.predict(x)[0]
            corrected.append(corrected_value)
        else:
            corrected.append(row[predicted_col])
    return corrected

In [ ]:
results = []
#find optimum no of bins
for n_bins in range(2, 11):
    tree = DecisionTreeRegressor(max_leaf_nodes=n_bins)
    tree.fit(df_train[['Actual']], df_train['RB'])

    thresholds = sorted(tree.tree_.threshold[tree.tree_.threshold > 0])
    bin_edges = [df_train['Actual'].min()] + thresholds + [df_train['Actual'].max()]

    df_train['TreeBin'] = pd.cut(df_train['Actual'], bins=bin_edges, include_lowest=True)
    df_val['TreeBin'] = pd.cut(df_val['Actual'], bins=bin_edges, include_lowest=True)

    lr_models = {}
    for label in df_train['TreeBin'].cat.categories:
        subset = df_train[df_train['TreeBin'] == label]
        if not subset.empty:
            X = subset[[predicted_col]]
            y = subset[observed_col]
            lr_models[label] = LinearRegression().fit(X, y)

    df_val['Corrected_Predicted'] = apply_tree_correction(df_val, lr_models)
    current_nse = nse(df_val['Corrected_Predicted'], df_val[observed_col])
    results.append({"Bins": n_bins, "NSE": round(current_nse, 4), "Breakpoints": np.round(bin_edges, 2).tolist()})

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# Step 2: Fit decision tree to detect change points in RB vs Observed
tree = DecisionTreeRegressor(max_leaf_nodes=5) 
tree.fit(df_train[['Actual']], df_train['RB'])

In [ ]:
# Step 3: bin assignment
thresholds = sorted(tree.tree_.threshold[tree.tree_.threshold > 0])
bin_edges = [df_train['Actual'].min()] + thresholds + [df_train['Actual'].max()]

df_train['TreeBin'] = pd.cut(df_train['Actual'], bins=bin_edges, include_lowest=True)
df_val['TreeBin'] = pd.cut(df_val['Actual'], bins=bin_edges, include_lowest=True)


In [ ]:
# Step 4: Train LR per bin
lr_models = {}
for label in df_train['TreeBin'].cat.categories:
    subset = df_train[df_train['TreeBin'] == label]
    if not subset.empty:
        X = subset[[predicted_col]]
        y = subset[observed_col]
        model = LinearRegression().fit(X, y)
        lr_models[label] = model

In [ ]:
# Step 5: Apply correction
def apply_tree_correction(df, models_dict):
    corrected = []
    for _, row in df.iterrows():
        bin_label = row['TreeBin']
        if pd.notnull(bin_label) and bin_label in models_dict:
            model = models_dict[bin_label]
            x = pd.DataFrame({predicted_col: [row[predicted_col]]})
            corrected_value = model.predict(x)[0]
            corrected.append(corrected_value)
        else:
            corrected.append(row[predicted_col])
    return corrected

df_val['Corrected_Predicted'] = apply_tree_correction(df_val, lr_models)

In [ ]:
# Step 6: Compute NSE
nse_original = nse(df_val[predicted_col], df_val[observed_col])
nse_corrected = nse(df_val['Corrected_Predicted'], df_val[observed_col])
